# 05 - Feature Engineering

### Objetivo
Construir todas las variables predictivas (features) necesarias para los modelos de forecasting a partir de los hallazgos del análisis exploratorio.

### Contexto de negocio
DSMarket necesita predecir ventas a nivel tienda-producto con un horizonte de 28 días (4 semanas). Para alimentar los modelos, crearemos features que capturen:
- Patrones temporales (estacionalidad, tendencia)
- Comportamiento histórico (lags, rolling statistics)
- Efectos de eventos y temporadas
- Información de precios

### Decisión clave: Agregación semanal
Trabajaremos a **nivel semanal** en lugar de diario porque:
- Reduce el ruido de las fluctuaciones diarias
- Minimiza el problema del 68% de ceros en datos diarios
- Simplifica el modelo (4 predicciones vs 28)
- El horizonte de 28 días = 4 semanas encaja perfectamente

### Qué conseguiremos
- Dataset listo para modelado con todas las features
- Estructura: una fila por combinación yearweek × item × store
- Features híbridas: temporales compartidas + lags/rolling por serie

## 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


## 2. Carga de datos procesados

In [2]:
# Cargar dataset procesado
df = pd.read_csv('/Users/anapeyra/Desktop/NUCLIO/Entregables/TFM/dsmarket-forecasting/data_dsmarket/processed/sales_processed.csv',
                 low_memory=False)

# Convertir fecha
df['date'] = pd.to_datetime(df['date'])

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Rango de fechas: {df['date'].min().date()} a {df['date'].max().date()}")
print(f"\nColumnas disponibles:")
print(df.columns.tolist())

Dataset cargado: 58,327,370 filas x 17 columnas
Rango de fechas: 2011-01-29 a 2016-04-24

Columnas disponibles:
['id', 'item', 'category', 'department', 'store', 'store_code', 'region', 'd', 'sales', 'date', 'weekday', 'weekday_int', 'event', 'year', 'week', 'yearweek', 'sell_price']


In [3]:
# Verificar estructura de yearweek
print(f"\nValores únicos de yearweek: {df['yearweek'].nunique()}")
print(f"Ejemplo de yearweek: {df['yearweek'].dropna().head().tolist()}")
print(f"Nulls en yearweek: {df['yearweek'].isna().sum():,}")


Valores únicos de yearweek: 275
Ejemplo de yearweek: [201104.0, 201104.0, 201104.0, 201104.0, 201104.0]
Nulls en yearweek: 0


## 3. Agregación a nivel semanal

Agrupamos las ventas diarias por semana para cada combinación producto-tienda.

In [4]:
# Crear identificador de serie (producto-tienda)
df['serie_id'] = df['item'] + '_' + df['store_code']

# Agregar a nivel semanal
df_weekly = df.groupby(['yearweek', 'serie_id', 'item', 'store', 'store_code', 
                        'category', 'department', 'region']).agg({
    'sales': 'sum',                    # Suma de ventas de la semana
    'sell_price': 'mean',              # Precio promedio de la semana
    'date': 'min',                     # Primera fecha de la semana (para referencia)
    'event': lambda x: x.dropna().tolist()  # Lista de eventos en la semana
}).reset_index()

# Renombrar columnas
df_weekly.rename(columns={'date': 'week_start', 'event': 'events_in_week'}, inplace=True)

print(f"Dataset semanal: {df_weekly.shape[0]:,} filas x {df_weekly.shape[1]} columnas")
print(f"Semanas únicas: {df_weekly['yearweek'].nunique()}")
print(f"Series únicas: {df_weekly['serie_id'].nunique()}")

Dataset semanal: 8,384,750 filas x 12 columnas
Semanas únicas: 275
Series únicas: 30490


In [5]:
# Verificar estructura
print("\nPrimeras filas del dataset semanal:")
df_weekly.head(10)


Primeras filas del dataset semanal:


,yearweek,serie_id,item,store,store_code,category,department,region,sales,sell_price,week_start,events_in_week
0,201104.0,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,South_End,BOS_1,ACCESORIES,ACCESORIES_1,Boston,0,NaN,2011-01-29,[]
1,201104.0,ACCESORIES_1_001_BOS_2,ACCESORIES_1_001,Roxbury,BOS_2,ACCESORIES,ACCESORIES_1,Boston,0,NaN,2011-01-29,[]
2,201104.0,ACCESORIES_1_001_BOS_3,ACCESORIES_1_001,Back_Bay,BOS_3,ACCESORIES,ACCESORIES_1,Boston,0,NaN,2011-01-29,[]
3,201104.0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,Greenwich_Village,NYC_1,ACCESORIES,ACCESORIES_1,New York,0,NaN,2011-01-29,[]
4,201104.0,ACCESORIES_1_001_NYC_2,ACCESORIES_1_001,Harlem,NYC_2,ACCESORIES,ACCESORIES_1,New York,0,NaN,2011-01-29,[]
5,201104.0,ACCESORIES_1_001_NYC_3,ACCESORIES_1_001,Tribeca,NYC_3,ACCESORIES,ACCESORIES_1,New York,0,NaN,2011-01-29,[]
6,201104.0,ACCESORIES_1_001_NYC_4,ACCESORIES_1_001,Brooklyn,NYC_4,ACCESORIES,ACCESORIES_1,New York,0,NaN,2011-01-29,[]
7,201104.0,ACCESORIES_1_001_PHI_1,ACCESORIES_1_001,Midtown_Village,PHI_1,ACCESORIES,ACCESORIES_1,Philadelphia,0,NaN,2011-01-29,[]
8,201104.0,ACCESORIES_1_001_PHI_2,ACCESORIES_1_001,Yorktown,PHI_2,ACCESORIES,ACCESORIES_1,Philadelphia,0,NaN,2011-01-29,[]
9,201104.0,ACCESORIES_1_001_PHI_3,ACCESORIES_1_001,Queen_Village,PHI_3,ACCESORIES,ACCESORIES_1,Philadelphia,0,NaN,2011-01-29,[]


In [6]:
# Estadísticas de ventas semanales
print("\nEstadísticas de ventas semanales:")
print(df_weekly['sales'].describe().round(2))

# Porcentaje de ceros
pct_zeros = (df_weekly['sales'] == 0).sum() / len(df_weekly) * 100
print(f"\nPorcentaje de semanas con cero ventas: {pct_zeros:.1f}%")


Estadísticas de ventas semanales:
count    8384750.00
mean           7.84
std           23.63
min            0.00
25%            0.00
50%            2.00
75%            7.00
max         3976.00
Name: sales, dtype: float64

Porcentaje de semanas con cero ventas: 40.2%


## 4. Features temporales

Extraemos componentes de tiempo de la semana (compartidas para todas las series).

In [10]:
# Verificar formato de yearweek
print("Ejemplos de yearweek:")
print(df_weekly['yearweek'].head(20))
print(f"\nTipo de dato: {df_weekly['yearweek'].dtype}")
print(f"Valores únicos (muestra): {df_weekly['yearweek'].dropna().unique()[:10]}")

Ejemplos de yearweek:
0     201104.0
1     201104.0
2     201104.0
3     201104.0
4     201104.0
5     201104.0
6     201104.0
7     201104.0
8     201104.0
9     201104.0
10    201104.0
11    201104.0
12    201104.0
13    201104.0
14    201104.0
15    201104.0
16    201104.0
17    201104.0
18    201104.0
19    201104.0
Name: yearweek, dtype: float64

Tipo de dato: float64
Valores únicos (muestra): [201104. 201105. 201106. 201107. 201108. 201109. 201110. 201111. 201112.
 201113.]


In [11]:
# Convertir yearweek a entero primero, luego extraer año y semana
df_weekly['yearweek_int'] = df_weekly['yearweek'].astype(int)

# Extraer año (primeros 4 dígitos) y semana (últimos 2 dígitos)
df_weekly['year'] = df_weekly['yearweek_int'] // 100
df_weekly['week'] = df_weekly['yearweek_int'] % 100

# Mes aproximado (basado en la fecha de inicio de semana)
df_weekly['month'] = df_weekly['week_start'].dt.month

# Trimestre
df_weekly['quarter'] = df_weekly['week_start'].dt.quarter

# Eliminar columna auxiliar
df_weekly.drop(columns=['yearweek_int'], inplace=True)

# Verificar
print("Features temporales creadas:")
print(df_weekly[['yearweek', 'year', 'week', 'month', 'quarter']].head(10))

Features temporales creadas:
   yearweek  year  week  month  quarter
0  201104.0  2011     4      1        1
1  201104.0  2011     4      1        1
2  201104.0  2011     4      1        1
3  201104.0  2011     4      1        1
4  201104.0  2011     4      1        1
5  201104.0  2011     4      1        1
6  201104.0  2011     4      1        1
7  201104.0  2011     4      1        1
8  201104.0  2011     4      1        1
9  201104.0  2011     4      1        1


In [12]:
# Feature de tendencia: semanas desde el inicio del dataset
fecha_inicio = df_weekly['week_start'].min()
df_weekly['weeks_since_start'] = ((df_weekly['week_start'] - fecha_inicio).dt.days / 7).astype(int)

print(f"\nFecha de inicio: {fecha_inicio.date()}")
print(f"Rango de weeks_since_start: {df_weekly['weeks_since_start'].min()} a {df_weekly['weeks_since_start'].max()}")


Fecha de inicio: 2011-01-29
Rango de weeks_since_start: 0 a 272


## 5. Features de lags (por serie)

Creamos variables con valores de ventas de semanas anteriores. Esto es crítico para forecasting.

**Lags a crear:**
- `lag_1`: Semana anterior (correlación más directa)
- `lag_2`: Hace 2 semanas
- `lag_4`: Hace 4 semanas (mismo punto del mes)
- `lag_52`: Hace 52 semanas (mismo punto del año anterior)

In [13]:
# Ordenar por serie y tiempo
df_weekly = df_weekly.sort_values(['serie_id', 'yearweek']).reset_index(drop=True)

# Crear lags por serie
lags = [1, 2, 4, 52]

for lag in lags:
    col_name = f'lag_{lag}'
    df_weekly[col_name] = df_weekly.groupby('serie_id')['sales'].shift(lag)
    print(f"Creado: {col_name}")

# Verificar
print("\nEjemplo de lags para una serie:")
ejemplo_serie = df_weekly[df_weekly['serie_id'] == df_weekly['serie_id'].iloc[0]]
print(ejemplo_serie[['yearweek', 'sales', 'lag_1', 'lag_2', 'lag_4', 'lag_52']].head(55).tail(10))

Creado: lag_1
Creado: lag_2
Creado: lag_4
Creado: lag_52

Ejemplo de lags para una serie:
    yearweek  sales  lag_1  lag_2  lag_4  lag_52
45  201149.0      0    0.0    0.0    0.0     NaN
46  201150.0      0    0.0    0.0    0.0     NaN
47  201151.0      0    0.0    0.0    0.0     NaN
48  201152.0      0    0.0    0.0    0.0     NaN
49  201201.0      0    0.0    0.0    0.0     NaN
50  201202.0      0    0.0    0.0    0.0     NaN
51  201203.0      0    0.0    0.0    0.0     NaN
52  201204.0      0    0.0    0.0    0.0     0.0
53  201205.0      0    0.0    0.0    0.0     0.0
54  201206.0      0    0.0    0.0    0.0     0.0


In [14]:
# Verificar nulls en lags (esperados al inicio de cada serie)
print("\nNulls en features de lag:")
for lag in lags:
    col_name = f'lag_{lag}'
    nulls = df_weekly[col_name].isna().sum()
    pct = nulls / len(df_weekly) * 100
    print(f"  {col_name}: {nulls:,} nulls ({pct:.1f}%)")


Nulls en features de lag:
  lag_1: 30,490 nulls (0.4%)
  lag_2: 60,980 nulls (0.7%)
  lag_4: 121,960 nulls (1.5%)
  lag_52: 1,585,480 nulls (18.9%)


## 6. Features de rolling statistics (por serie)

Estadísticas móviles que capturan el comportamiento reciente de cada serie.

**Rolling features:**
- `rolling_mean_4`: Media de las últimas 4 semanas
- `rolling_mean_12`: Media de las últimas 12 semanas (trimestre)
- `rolling_std_4`: Desviación estándar de las últimas 4 semanas (volatilidad)

In [15]:
# Rolling statistics por serie
# Usamos shift(1) para no incluir la semana actual (evitar data leakage)

# Rolling mean 4 semanas
df_weekly['rolling_mean_4'] = df_weekly.groupby('serie_id')['sales'].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=1).mean()
)

# Rolling mean 12 semanas
df_weekly['rolling_mean_12'] = df_weekly.groupby('serie_id')['sales'].transform(
    lambda x: x.shift(1).rolling(window=12, min_periods=1).mean()
)

# Rolling std 4 semanas
df_weekly['rolling_std_4'] = df_weekly.groupby('serie_id')['sales'].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=1).std()
)

print("Features de rolling creadas: rolling_mean_4, rolling_mean_12, rolling_std_4")

Features de rolling creadas: rolling_mean_4, rolling_mean_12, rolling_std_4


In [17]:
# Verificar rolling features
ejemplo_serie = df_weekly[df_weekly['serie_id'] == df_weekly['serie_id'].iloc[0]].copy()

print("\nEjemplo de rolling features para una serie:")
print(ejemplo_serie[['yearweek', 'sales', 'rolling_mean_4', 'rolling_mean_12', 'rolling_std_4']].head(15))


Ejemplo de rolling features para una serie:
    yearweek  sales  rolling_mean_4  rolling_mean_12  rolling_std_4
0   201104.0      0             NaN              NaN            NaN
1   201105.0      0             0.0              0.0            NaN
2   201106.0      0             0.0              0.0            0.0
3   201107.0      0             0.0              0.0            0.0
4   201108.0      0             0.0              0.0            0.0
5   201109.0      0             0.0              0.0            0.0
6   201110.0      0             0.0              0.0            0.0
7   201111.0      0             0.0              0.0            0.0
8   201112.0      0             0.0              0.0            0.0
9   201113.0      0             0.0              0.0            0.0
10  201114.0      0             0.0              0.0            0.0
11  201115.0      0             0.0              0.0            0.0
12  201116.0      0             0.0              0.0            0.0
13 

## 7. Features de eventos

Convertimos la lista de eventos por semana en features numéricas útiles.

In [18]:
# Número de eventos en la semana
df_weekly['n_events'] = df_weekly['events_in_week'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Tiene algún evento
df_weekly['has_event'] = (df_weekly['n_events'] > 0).astype(int)

print("Distribución de eventos por semana:")
print(df_weekly['n_events'].value_counts().sort_index())

Distribución de eventos por semana:
n_events
0    5671140
1    2439200
2     274410
Name: count, dtype: int64


In [19]:
# Función para verificar si un evento específico está en la semana
def has_specific_event(events_list, event_name):
    if isinstance(events_list, list):
        return int(event_name in events_list)
    return 0

# Eventos críticos identificados en el EDA
eventos_criticos = ['Christmas', 'Thanksgiving', 'Black_Friday', 'SuperBowl', 
                    'Labor_Day', 'Easter', 'Cyber_Monday']

for evento in eventos_criticos:
    col_name = f'has_{evento.lower()}'
    df_weekly[col_name] = df_weekly['events_in_week'].apply(
        lambda x: has_specific_event(x, evento)
    )
    n_semanas = df_weekly[col_name].sum()
    print(f"Creado: {col_name} ({n_semanas} semanas)")

Creado: has_christmas (152450 semanas)
Creado: has_thanksgiving (152450 semanas)
Creado: has_black_friday (152450 semanas)
Creado: has_superbowl (182940 semanas)
Creado: has_labor_day (152450 semanas)
Creado: has_easter (152450 semanas)
Creado: has_cyber_monday (152450 semanas)


In [20]:
# Feature especial: tienda cerrada (Christmas)
df_weekly['is_store_closed'] = df_weekly['has_christmas']

print(f"\nSemanas con tienda cerrada: {df_weekly['is_store_closed'].sum()}")


Semanas con tienda cerrada: 152450


## 8. Features de temporadas

Asignamos la temporada comercial predominante de cada semana.

In [21]:
# Función para asignar temporada según mes y día
def assign_season(date):
    month = date.month
    day = date.day
    
    # Winter Holidays: 15 nov - 31 dic
    if (month == 11 and day >= 15) or month == 12:
        return 'Winter_Holidays'
    # Spring Break: 1 mar - 15 abr
    elif (month == 3) or (month == 4 and day <= 15):
        return 'Spring_Break'
    # Summer Vacation: 1 jun - 14 jul
    elif (month == 6) or (month == 7 and day <= 14):
        return 'Summer_Vacation'
    # Back to School: 15 jul - 15 sep
    elif (month == 7 and day >= 15) or month == 8 or (month == 9 and day <= 15):
        return 'Back_to_School'
    else:
        return 'Regular'

# Aplicar a la fecha de inicio de semana
df_weekly['season'] = df_weekly['week_start'].apply(assign_season)

print("Distribución de semanas por temporada:")
print(df_weekly['season'].value_counts())

Distribución de semanas por temporada:
season
Regular            3933210
Back_to_School     1372050
Spring_Break       1189110
Summer_Vacation     975680
Winter_Holidays     914700
Name: count, dtype: int64


In [22]:
# Variables dummy para temporadas
season_dummies = pd.get_dummies(df_weekly['season'], prefix='season')
df_weekly = pd.concat([df_weekly, season_dummies], axis=1)

print("\nVariables dummy de temporada creadas:")
print([col for col in df_weekly.columns if col.startswith('season_')])


Variables dummy de temporada creadas:
['season_Back_to_School', 'season_Regular', 'season_Spring_Break', 'season_Summer_Vacation', 'season_Winter_Holidays']


## 9. Features de precio

Variables relacionadas con el precio de venta.

In [23]:
# Precio promedio por item (para comparar)
precio_promedio_item = df_weekly.groupby('item')['sell_price'].transform('mean')

# Precio relativo (vs promedio del producto)
df_weekly['price_vs_avg'] = df_weekly['sell_price'] / precio_promedio_item

# Cambio de precio vs semana anterior
df_weekly['price_change'] = df_weekly.groupby('serie_id')['sell_price'].pct_change()

# Reemplazar infinitos y nulls en price_change
df_weekly['price_change'] = df_weekly['price_change'].replace([np.inf, -np.inf], np.nan)
df_weekly['price_change'] = df_weekly['price_change'].fillna(0)

print("Features de precio creadas:")
print(df_weekly[['sell_price', 'price_vs_avg', 'price_change']].describe().round(3))

Features de precio creadas:
        sell_price  price_vs_avg  price_change
count  6544817.000   6544817.000   8384750.000
mean         5.515         1.000         0.001
std          4.381         0.058         0.406
min          0.012         0.001        -0.999
25%          2.620         0.990         0.000
50%          4.200         1.001         0.000
75%          7.176         1.016         0.000
max        134.150         8.354       897.000


## 10. Verificación y limpieza final

In [24]:
# Revisar todas las columnas del dataset final
print("Columnas del dataset final:")
print(df_weekly.columns.tolist())
print(f"\nTotal columnas: {len(df_weekly.columns)}")

Columnas del dataset final:
['yearweek', 'serie_id', 'item', 'store', 'store_code', 'category', 'department', 'region', 'sales', 'sell_price', 'week_start', 'events_in_week', 'year', 'month', 'quarter', 'week', 'weeks_since_start', 'lag_1', 'lag_2', 'lag_4', 'lag_52', 'rolling_mean_4', 'rolling_mean_12', 'rolling_std_4', 'n_events', 'has_event', 'has_christmas', 'has_thanksgiving', 'has_black_friday', 'has_superbowl', 'has_labor_day', 'has_easter', 'has_cyber_monday', 'is_store_closed', 'season', 'season_Back_to_School', 'season_Regular', 'season_Spring_Break', 'season_Summer_Vacation', 'season_Winter_Holidays', 'price_vs_avg', 'price_change']

Total columnas: 42


In [25]:
# Verificar nulls en todas las columnas
print("\nNulls por columna:")
nulls = df_weekly.isnull().sum()
nulls_pct = (nulls / len(df_weekly) * 100).round(2)
null_summary = pd.DataFrame({'nulls': nulls, 'pct': nulls_pct})
null_summary = null_summary[null_summary['nulls'] > 0].sort_values('nulls', ascending=False)
print(null_summary)


Nulls por columna:
                   nulls    pct
sell_price       1839933  21.94
price_vs_avg     1839933  21.94
lag_52           1585480  18.91
lag_4             121960   1.45
lag_2              60980   0.73
rolling_std_4      60980   0.73
lag_1              30490   0.36
rolling_mean_4     30490   0.36
rolling_mean_12    30490   0.36


In [27]:
# Decisión sobre nulls en lags
print("\n=== TRATAMIENTO DE NULLS ===")
print("\nLos nulls en lags son esperados (primeras semanas de cada serie).")
print("Opciones:")
print("  1. Eliminar filas con nulls (perdemos datos iniciales)")
print("  2. Rellenar con 0 (asume que no había ventas)")
print("  3. Rellenar con la media de la serie")
print("  4. Mantener nulls (algunos modelos los manejan)")

# Calcular cuántas filas perderíamos
filas_con_null_lags = df_weekly[df_weekly[['lag_1', 'lag_52']].isna().any(axis=1)]
print(f"\nFilas con al menos un null en lags: {len(filas_con_null_lags):,} ({len(filas_con_null_lags)/len(df_weekly)*100:.1f}%)")

# Nulls en precio
filas_sin_precio = df_weekly['sell_price'].isna().sum()
print(f"Filas sin precio (sell_price): {filas_sin_precio:,} ({filas_sin_precio/len(df_weekly)*100:.1f}%)")


=== TRATAMIENTO DE NULLS ===

Los nulls en lags son esperados (primeras semanas de cada serie).
Opciones:
  1. Eliminar filas con nulls (perdemos datos iniciales)
  2. Rellenar con 0 (asume que no había ventas)
  3. Rellenar con la media de la serie
  4. Mantener nulls (algunos modelos los manejan)

Filas con al menos un null en lags: 1,585,480 (18.9%)
Filas sin precio (sell_price): 1,839,933 (21.9%)


In [28]:
# Rellenar nulls en lags con 0 (conservador, asume sin historial)
lag_cols = ['lag_1', 'lag_2', 'lag_4', 'lag_52']
rolling_cols = ['rolling_mean_4', 'rolling_mean_12', 'rolling_std_4']

for col in lag_cols + rolling_cols:
    df_weekly[col] = df_weekly[col].fillna(0)

# Rellenar nulls en precio con la media del item
df_weekly['sell_price'] = df_weekly.groupby('item')['sell_price'].transform(
    lambda x: x.fillna(x.mean())
)
df_weekly['price_vs_avg'] = df_weekly['price_vs_avg'].fillna(1)

print("Nulls rellenados en lags, rolling y precio")

Nulls rellenados en lags, rolling y precio


In [29]:
# Verificación final de nulls
print("\nVerificación final de nulls:")
nulls_final = df_weekly.isnull().sum()
nulls_final = nulls_final[nulls_final > 0]
if len(nulls_final) == 0:
    print("✓ No hay nulls en el dataset")
else:
    print(nulls_final)


Verificación final de nulls:
✓ No hay nulls en el dataset


In [30]:
# Eliminar columnas auxiliares que no necesitamos para modelado
cols_to_drop = ['events_in_week', 'week_start']
df_weekly = df_weekly.drop(columns=cols_to_drop)

print(f"Dataset final: {df_weekly.shape[0]:,} filas x {df_weekly.shape[1]} columnas")

Dataset final: 8,384,750 filas x 40 columnas


In [31]:
# Mostrar estructura final
print("\n=== ESTRUCTURA FINAL DEL DATASET ===\n")

# Agrupar columnas por tipo
cols_id = ['yearweek', 'serie_id', 'item', 'store', 'store_code', 'category', 'department', 'region']
cols_target = ['sales']
cols_temporal = ['year', 'week', 'month', 'quarter', 'weeks_since_start']
cols_lags = ['lag_1', 'lag_2', 'lag_4', 'lag_52']
cols_rolling = ['rolling_mean_4', 'rolling_mean_12', 'rolling_std_4']
cols_eventos = ['n_events', 'has_event', 'has_christmas', 'has_thanksgiving', 
                'has_black_friday', 'has_superbowl', 'has_labor_day', 'has_easter', 
                'has_cyber_monday', 'is_store_closed']
cols_season = ['season'] + [c for c in df_weekly.columns if c.startswith('season_')]
cols_precio = ['sell_price', 'price_vs_avg', 'price_change']

print(f"Identificadores ({len(cols_id)}): {cols_id}")
print(f"\nTarget ({len(cols_target)}): {cols_target}")
print(f"\nFeatures temporales ({len(cols_temporal)}): {cols_temporal}")
print(f"\nFeatures de lags ({len(cols_lags)}): {cols_lags}")
print(f"\nFeatures de rolling ({len(cols_rolling)}): {cols_rolling}")
print(f"\nFeatures de eventos ({len(cols_eventos)}): {cols_eventos}")
print(f"\nFeatures de temporada ({len(cols_season)}): {cols_season}")
print(f"\nFeatures de precio ({len(cols_precio)}): {cols_precio}")


=== ESTRUCTURA FINAL DEL DATASET ===

Identificadores (8): ['yearweek', 'serie_id', 'item', 'store', 'store_code', 'category', 'department', 'region']

Target (1): ['sales']

Features temporales (5): ['year', 'week', 'month', 'quarter', 'weeks_since_start']

Features de lags (4): ['lag_1', 'lag_2', 'lag_4', 'lag_52']

Features de rolling (3): ['rolling_mean_4', 'rolling_mean_12', 'rolling_std_4']

Features de eventos (10): ['n_events', 'has_event', 'has_christmas', 'has_thanksgiving', 'has_black_friday', 'has_superbowl', 'has_labor_day', 'has_easter', 'has_cyber_monday', 'is_store_closed']

Features de temporada (6): ['season', 'season_Back_to_School', 'season_Regular', 'season_Spring_Break', 'season_Summer_Vacation', 'season_Winter_Holidays']

Features de precio (3): ['sell_price', 'price_vs_avg', 'price_change']


In [32]:
# Guardar dataset final
output_path = '/Users/anapeyra/Desktop/NUCLIO/Entregables/TFM/dsmarket-forecasting/data_dsmarket/processed/sales_weekly_features.csv'
df_weekly.to_csv(output_path, index=False)

print(f"\n✓ Dataset guardado en: {output_path}")
print(f"  Tamaño: {df_weekly.shape[0]:,} filas x {df_weekly.shape[1]} columnas")


✓ Dataset guardado en: /Users/anapeyra/Desktop/NUCLIO/Entregables/TFM/dsmarket-forecasting/data_dsmarket/processed/sales_weekly_features.csv
  Tamaño: 8,384,750 filas x 40 columnas


## 11. Conclusiones

### Transformación realizada

| Métrica | Datos diarios | Datos semanales | Cambio |
|---------|---------------|-----------------|--------|
| Filas | 58,327,370 | 8,384,750 | -86% |
| Granularidad | día × item × store | semana × item × store | Agregación |
| % ceros | 68.2% | 40.2% | -28 pp |
| Semanas/días | 1,913 días | 275 semanas | - |

**Beneficio clave:** La agregación semanal reduce el ruido y el problema de datos sparse, manteniendo la información relevante para predicciones de 28 días (4 semanas).

---

### Features creadas (27 variables predictivas)

| Grupo | # | Variables | Propósito |
|-------|---|-----------|-----------|
| **Temporales** | 5 | year, week, month, quarter, weeks_since_start | Capturar estacionalidad y tendencia |
| **Lags** | 4 | lag_1, lag_2, lag_4, lag_52 | Autocorrelación (semanas anteriores) |
| **Rolling** | 3 | rolling_mean_4, rolling_mean_12, rolling_std_4 | Comportamiento reciente y volatilidad |
| **Eventos** | 10 | has_christmas, has_superbowl, etc. | Efectos de días especiales |
| **Temporadas** | 6 | season_*, is_store_closed | Periodos comerciales |
| **Precio** | 3 | sell_price, price_vs_avg, price_change | Efecto del precio |

---

### Tratamiento de nulls

| Tipo | Causa | Solución |
|------|-------|----------|
| Lags (18.9%) | Primeras semanas sin historial | Rellenado con 0 |
| Rolling (0.4-0.7%) | Primeras semanas | Rellenado con 0 |
| Precio (21.9%) | Semanas sin información de precio | Rellenado con media del item |

---

### Dataset final

- **Archivo:** `data_dsmarket/processed/sales_weekly_features.csv`
- **Dimensiones:** 8,384,750 filas × 40 columnas
- **Series:** 30,490 combinaciones item × store
- **Horizonte temporal:** 275 semanas (2011-W04 a 2016-W17)
- **Sin nulls:** ✓

---

### Estructura para modelado
```
Identificadores (8):  yearweek, serie_id, item, store, store_code, category, department, region
Target (1):           sales
Features (31):        temporales + lags + rolling + eventos + temporadas + precio
```

---